In [ ]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

In [ ]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql-ctp21;TrustServerCertificate=True;Integrated Security=True"

In [12]:
SELECT @@VERSION

(1 row affected)

(No column name)
Microsoft SQL Server 2025 (CTP2.1) - 17.0.800.3 (X64) Jun 12 2025 14:47:57 Copyright (C) 2025 Microsoft Corporation Enterprise Evaluation Edition (64-bit) on Windows Server 2022 Standard 10.0 <X64> (Build 20348: ) (Hypervisor)


In [13]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='FileData')BEGIN
    ALTER DATABASE FileData SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE FileData;
END
GO
CREATE DATABASE FileData
GO
USE FileData
GO
create master key encryption by password = 'MyTest!Mast3rP4ss'

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

In [14]:
SELECT SERVERPROPERTY('IsPolyBaseInstalled') AS IsPolyBaseInstalled;

SELECT name, value, value_in_use
FROM sys.configurations
WHERE name LIKE 'polybase enabled';

(1 row affected)

(1 row affected)

IsPolyBaseInstalled
<null>


name,value,value_in_use
polybase enabled,<null>,<null>


In [15]:
SELECT * FROM OPENROWSET(
   BULK 'C:\DATA\SampleData.txt',
   SINGLE_CLOB
) AS DATA;

(1 row affected)

BulkColumn
"ID,Name,Email,Country,Age 1,Alice Smith,alice@example.com,USA,30 2,Bob Johnson,bob@example.com,Canada,45 3,Carla Brown,carla@example.com,UK,27 4,David Wilson,david@example.com,Australia,38 5,Emma Davis,emma@example.com,Germany,32 6,Frank Miller,frank@example.com,France,41 7,Grace Lee,grace@example.com,South Korea,29 8,Henry Moore,henry@example.com,India,35 9,Ivy Taylor,ivy@example.com,Japan,26 10,Jack White,jack@example.com,Brazil,33"


In [16]:
CREATE TABLE SampleData (
    ID INT,
    Name NVARCHAR(100),
    Email NVARCHAR(100),
    Country NVARCHAR(50),
    Age INT
);


Commands completed successfully.

In [17]:
BULK INSERT SampleData
FROM 'C:\data\SampleData.txt'
WITH (
    FIRSTROW = 2,
    FIELDTERMINATOR = ',',
    ROWTERMINATOR = '\n',
    TABLOCK
);

(10 rows affected)

In [18]:
SELECT * FROM SampleData

(10 rows affected)

ID,Name,Email,Country,Age
1,Alice Smith,alice@example.com,USA,30
2,Bob Johnson,bob@example.com,Canada,45
3,Carla Brown,carla@example.com,UK,27
4,David Wilson,david@example.com,Australia,38
5,Emma Davis,emma@example.com,Germany,32
6,Frank Miller,frank@example.com,France,41
7,Grace Lee,grace@example.com,South Korea,29
8,Henry Moore,henry@example.com,India,35
9,Ivy Taylor,ivy@example.com,Japan,26
10,Jack White,jack@example.com,Brazil,33


In [19]:
CREATE TABLE dbo.SalesData (
    SaleID INT IDENTITY(1,1) PRIMARY KEY,
    SaleDate DATE,
    CustomerID INT,
    ProductID INT,
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount AS (Quantity * UnitPrice) PERSISTED
);

;WITH Tally AS (
    SELECT TOP (10000000) ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS n
    FROM sys.all_objects a CROSS JOIN sys.all_objects b
),
RandomSales AS (
    SELECT
        DATEADD(DAY, -ABS(CHECKSUM(NEWID()) % 365), GETDATE()) AS SaleDate,
        ABS(CHECKSUM(NEWID())) % 1000 + 1 AS CustomerID,
        ABS(CHECKSUM(NEWID())) % 100 + 1 AS ProductID,
        ABS(CHECKSUM(NEWID())) % 10 + 1 AS Quantity,
        CAST(ABS(CHECKSUM(NEWID())) % 5000 + 100 AS DECIMAL(10,2)) / 100.0 AS UnitPrice
    FROM Tally
)
INSERT INTO dbo.SalesData (SaleDate, CustomerID, ProductID, Quantity, UnitPrice)
SELECT SaleDate, CustomerID, ProductID, Quantity, UnitPrice
FROM RandomSales;


(7601049 rows affected)

In [20]:
SELECT TOP 10 * FROM SalesData

(10 rows affected)

SaleID,SaleDate,CustomerID,ProductID,Quantity,UnitPrice,TotalAmount
1,2024-11-01 00:00:00Z,165,8,2,6.16,12.32
2,2025-02-16 00:00:00Z,91,27,4,41.03,164.12
3,2024-09-29 00:00:00Z,385,6,9,19.41,174.69
4,2025-02-12 00:00:00Z,246,36,3,37.68,113.04
5,2025-04-20 00:00:00Z,217,21,6,29.42,176.52
6,2025-05-10 00:00:00Z,583,73,9,2.90,26.10
7,2025-04-23 00:00:00Z,984,50,2,28.24,56.48
8,2025-05-10 00:00:00Z,175,74,2,11.27,22.54
9,2024-09-10 00:00:00Z,687,84,8,39.89,319.12
10,2025-04-22 00:00:00Z,450,89,5,12.77,63.85


In [21]:
sp_configure 'allow polybase export', 1
GO
RECONFIGURE

Configuration option 'allow polybase export' changed from 1 to 1. Run the RECONFIGURE statement to install.

Commands completed successfully.

In [22]:
CREATE EXTERNAL FILE FORMAT CsvFileFormat
WITH (
    FORMAT_TYPE = DELIMITEDTEXT,
    FORMAT_OPTIONS (
        FIELD_TERMINATOR = ',',
        STRING_DELIMITER = '"',
        FIRST_ROW = 2, 
        USE_TYPE_DEFAULT = TRUE
    )
);

Commands completed successfully.

In [23]:
CREATE EXTERNAL FILE FORMAT ParquetFileFormat
    WITH (FORMAT_TYPE = PARQUET);

Commands completed successfully.

In [24]:
CREATE DATABASE SCOPED CREDENTIAL s3 WITH IDENTITY = 'S3 Access Key', SECRET = 'minioadmin:minioadmin'

Commands completed successfully.

In [25]:
CREATE EXTERNAL DATA SOURCE s3
                WITH 
                (    LOCATION = 's3://storage.lab.bwdemo.io:9000/',
                     CREDENTIAL = s3
                )

Commands completed successfully.

In [26]:
select *
from openrowset(
        bulk '/salesdata/SampleData.txt',
        data_source = 's3',
        format = 'csv',
        firstrow = 2
) WITH (
    ID INT,
    Name NVARCHAR(100),
    Email NVARCHAR(100),
    Country NVARCHAR(50),
    Age INT
) as ro

(10 rows affected)

ID,Name,Email,Country,Age
1,Alice Smith,alice@example.com,USA,30
2,Bob Johnson,bob@example.com,Canada,45
3,Carla Brown,carla@example.com,UK,27
4,David Wilson,david@example.com,Australia,38
5,Emma Davis,emma@example.com,Germany,32
6,Frank Miller,frank@example.com,France,41
7,Grace Lee,grace@example.com,South Korea,29
8,Henry Moore,henry@example.com,India,35
9,Ivy Taylor,ivy@example.com,Japan,26
10,Jack White,jack@example.com,Brazil,33


In [27]:
CREATE EXTERNAL TABLE SalesData_S3_Parquet
    WITH (
            LOCATION = '/salesdata/Parquet',
            DATA_SOURCE = s3,
            FILE_FORMAT = ParquetFileFormat
            ) AS

SELECT * FROM SalesData 

(7601049 rows affected)

In [28]:
CREATE EXTERNAL TABLE SalesData_S3_CSV
    WITH (
            LOCATION = '/salesdata/CSV',
            DATA_SOURCE = s3,
            FILE_FORMAT = CSVFileFormat
            ) AS

SELECT * FROM SalesData 

(7601049 rows affected)

In [29]:
SELECT CustomerId,COUNT(*) FROM SalesData_S3_Parquet GROUP BY CustomerId

(1000 rows affected)

CustomerId,(No column name)
782,7588
850,7696
204,7551
391,7547
272,7613
765,7589
442,7526
757,7558
186,7646
154,7616


In [30]:
SELECT CustomerId,COUNT(*) FROM SalesData_S3_CSV GROUP BY CustomerId

(1000 rows affected)

CustomerId,(No column name)
18,7677
187,7649
255,7647
680,7611
476,7498
459,7667
970,7789
952,7580
663,7518
68,7640


In [ ]:
CREATE DATABASE SCOPED CREDENTIAL ADLS_SAS_Token
WITH IDENTITY = 'SHARED ACCESS SIGNATURE',
SECRET = '<SAS TOKEN>';

CREATE EXTERNAL DATA SOURCE ADLS_Archive
WITH (
    LOCATION = 'abs://archive@bwadlsarchive.blob.core.windows.net/',
    CREDENTIAL = ADLS_SAS_Token
);